**Setup**

In [ ]:
!pip install llama-cpp-python

In [ ]:
import os, random, sys
import numpy as np
from pathlib import Path

SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
random.seed(SEED)
np.random.seed(SEED)

try:
    import torch
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)
    try:
        torch.use_deterministic_algorithms(True, warn_only=True)
    except Exception:
        pass
except ImportError:
    pass

print("Seeded with", SEED)

# --- Path configuration: change only this block when migrating ---
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    # UPDATE THIS to your actual project path on Google Drive
    PROJECT_ROOT = Path('/content/drive/MyDrive/SRH/Thesis')
else:
    PROJECT_ROOT = Path('..').resolve()

# Derived paths (no need to edit these)
DATA_DIR       = PROJECT_ROOT / 'data'
DARPA_DIR      = DATA_DIR / 'darpa'
ATTCK_DIR      = DATA_DIR / 'attck'
RESULTS_DIR    = PROJECT_ROOT / 'results' / 'mistral-nemo-12b'
MODELS_DIR     = PROJECT_ROOT / 'models'
MODEL_FILENAME = 'Mistral-Nemo-Instruct-2407-Q5_K_M.gguf'
MODEL_PATH     = str(MODELS_DIR / MODEL_FILENAME)

print(f'Project root: {PROJECT_ROOT}')
print(f'Models:       {MODELS_DIR}')
print(f'Data (DARPA): {DARPA_DIR}')
print(f'Results:      {RESULTS_DIR}')


**1. Load CADETS E3**

In [ ]:
import json, glob, sys
from datetime import datetime, timezone
from collections import Counter

CADETS_PATTERN = str(DARPA_DIR / "ta1-cadets-e3-official*/ta1-cadets-e3-official*.json*")
CADETS_FILES = sorted(glob.glob(CADETS_PATTERN))
if not CADETS_FILES:
    raise FileNotFoundError(
        f'No CADETS data files found matching: {CADETS_PATTERN}\n'
        f'Checked in: {DARPA_DIR}')

P = "com.bbn.tc.schema.avro.cdm18."
print(f"{len(CADETS_FILES)} CADETS file(s):")
for f in CADETS_FILES:
    print(f"  {f}")

def utc_ns(y, mo, d, h, mi):
    return int(datetime(y, mo, d, h, mi, tzinfo=timezone.utc).timestamp() * 1e9)

WINDOWS = {
    "W1": {
        "start": utc_ns(2018,4,6,14,0), "end": utc_ns(2018,4,6,17,0),
        "ips": {"81.49.200.166","78.205.235.65","200.36.109.214","139.123.0.113",
                "152.111.159.139","154.143.113.18","61.167.39.128"},
        "paths": {"/tmp/vUgefal","/var/log/devc"},
        "gt_observable": {"T1190","T1071","T1105","T1059","T1055","T1222"},
        "gt_campaign":   {"T1190","T1071","T1105","T1059","T1055","T1222","T1046","T1068"},
    },
    "W2": {
        "start": utc_ns(2018,4,11,18,30), "end": utc_ns(2018,4,11,19,45),
        "ips": {"25.159.96.207","76.56.184.25","155.162.39.48","198.115.236.119"},
        "paths": {"sendmail","grain","/tmp/grain"},
        "gt_observable": {"T1190","T1071","T1105","T1059","T1055"},
        "gt_campaign":   {"T1190","T1071","T1105","T1059","T1055"},
    },
    "W3": {
        "start": utc_ns(2018,4,12,17,30), "end": utc_ns(2018,4,12,19,0),
        "ips": {"25.159.96.207","76.56.184.25","155.162.39.48","198.115.236.119",
                "53.158.101.118","98.15.44.232","192.113.144.28"},
        "paths": {"tmux-1002","minions","font","XIM","netlog","sendmail","main","/tmp/test"},
        "gt_observable": {"T1190","T1071","T1105","T1059","T1070.004","T1222"},
        "gt_campaign":   {"T1190","T1071","T1105","T1059","T1070.004","T1222","T1046","T1041","T1068"},
    },
    "W4": {
        "start": utc_ns(2018,4,13,12,30), "end": utc_ns(2018,4,13,14,0),
        "ips": {"25.159.96.207","76.56.184.25","155.162.39.48","198.115.236.119","53.158.101.118"},
        "paths": {"pEja72mA","eWq10bVcx","memhelp.so","eraseme","done.so"},
        "gt_observable": {"T1190","T1071","T1105","T1059","T1055","T1222"},
        "gt_campaign":   {"T1190","T1071","T1105","T1059","T1055","T1222","T1068"},
    },
}

for w in WINDOWS.values():
    w["gt_set"] = w["gt_observable"]

GLOBAL_START = min(w["start"] for w in WINDOWS.values())
GLOBAL_END   = max(w["end"] for w in WINDOWS.values())

def cdm_type(datum):
    return next(iter(datum)).replace(P, "")

subjects, files, netflows = {}, {}, {}
for path in CADETS_FILES:
    with open(path) as f:
        for line in f:
            try:
                d = json.loads(line)["datum"]
            except Exception:
                continue
            t = cdm_type(d)
            obj = d[P + t]
            u = obj.get("uuid")
            if t == "Subject":
                props = obj.get("properties", {}).get("map", {})
                subjects[u] = props.get("exec") or obj.get("cmdLine") or "process"
            elif t == "FileObject":
                files[u] = obj.get("type", "FILE")
            elif t == "NetFlowObject":
                netflows[u] = f'{obj.get("remoteAddress")}:{obj.get("remotePort")}'

print(f"Subjects {len(subjects)} | Files {len(files)} | NetFlows {len(netflows)}")

def which_window(ts):
    for wid, w in WINDOWS.items():
        if w["start"] <= ts <= w["end"]:
            return wid
    return None

events = []
for path in CADETS_FILES:
    with open(path) as f:
        for line in f:
            try:
                d = json.loads(line)["datum"]
            except Exception:
                continue
            if cdm_type(d) != "Event":
                continue
            e = d[P + "Event"]
            ts = e.get("timestampNanos", 0)
            if not (GLOBAL_START <= ts <= GLOBAL_END):
                continue
            wid = which_window(ts)
            if wid is None:
                continue
            e["_window"] = wid
            events.append(e)

events.sort(key=lambda e: (e.get("timestampNanos", 0), e.get("uuid", "")))
per_window = dict(Counter(e["_window"] for e in events))
print("Events per window:", per_window)
print("Total events:", len(events))


**2. Build alerts**

In [ ]:
SEMANTIC = {"EVENT_EXECUTE","EVENT_WRITE","EVENT_CREATE_OBJECT","EVENT_FORK",
            "EVENT_MODIFY_FILE_ATTRIBUTES","EVENT_MODIFY_PROCESS","EVENT_UNLINK",
            "EVENT_CHANGE_PRINCIPAL","EVENT_RENAME","EVENT_LINK","EVENT_LOGIN",
            "EVENT_MPROTECT","EVENT_TRUNCATE"}
TRAFFIC  = {"EVENT_CONNECT","EVENT_SENDTO","EVENT_RECVFROM","EVENT_ACCEPT",
            "EVENT_SENDMSG","EVENT_RECVMSG","EVENT_BIND"}
INBOUND  = {"EVENT_ACCEPT","EVENT_BIND"}

def event_targets(e):
    ips, paths = set(), set()
    for k in ("predicateObject","predicateObject2"):
        ref = e.get(k)
        if ref:
            nf = netflows.get(ref.get(P+"UUID"))
            if nf:
                ips.add(nf.split(":")[0])
    for k in ("predicateObjectPath","predicateObject2Path"):
        pth = e.get(k)
        if pth:
            s = pth.get("string") if isinstance(pth, dict) else pth
            if s:
                paths.add(s)
    return ips, paths

def path_matches_ioc(p, iocs):
    base = p.rsplit("/", 1)[-1]
    for ioc in iocs:
        ioc_base = ioc.rsplit("/", 1)[-1]
        if ioc.startswith("/"):
            if p == ioc or base == ioc_base:
                return ioc
        elif base == ioc_base:
            return ioc
    return None

def is_mal(wid, ips, paths):
    w = WINDOWS[wid]
    if ips & w["ips"]:
        return True
    return any(path_matches_ioc(p, w["paths"]) for p in paths)

VERB = {
    "EVENT_EXECUTE":"executed", "EVENT_WRITE":"wrote to", "EVENT_CREATE_OBJECT":"created",
    "EVENT_FORK":"forked", "EVENT_MODIFY_FILE_ATTRIBUTES":"changed permissions on",
    "EVENT_MODIFY_PROCESS":"modified process", "EVENT_UNLINK":"deleted",
    "EVENT_CHANGE_PRINCIPAL":"changed privilege via", "EVENT_RENAME":"renamed",
    "EVENT_LINK":"linked", "EVENT_LOGIN":"logged in via",
    "EVENT_MPROTECT":"changed memory protection on", "EVENT_TRUNCATE":"truncated",
}

def role_tag(target):
    t = str(target).lower()
    if t in ("a socket/pipe", "<unknown>", ""):
        return ""
    if "/tmp/" in t or t.startswith("tmp"):
        return " (file in temp directory)"
    if "/var/log" in t or "log" in t:
        return " (file under system log directory)"
    if t.endswith(".so") or "memhelp" in t:
        return " (shared library / loadable module)"
    if "sshd" in t or "ssh" in t:
        return " (SSH service process)"
    if "nginx" in t:
        return " (web server process)"
    if any(c.isdigit() for c in t) and "." in t and "/" not in t:
        return " (external network host)"
    return ""

def chan_phrase(c):
    if c.get("dir") == "inbound":
        return f'accepted inbound connection from {c["ip"]} ({c["count"]} packets)'
    return f'connected to {c["ip"]} ({c["count"]} packets)'

alerts = []
for wid in WINDOWS:
    sem, chan = [], {}
    for e in events:
        if e["_window"] != wid:
            continue
        et = e.get("type", "?")
        ips, paths = event_targets(e)
        mal = is_mal(wid, ips, paths)
        subj_ref = e.get("subject") or {}
        eprops = (e.get("properties") or {}).get("map") or {}
        proc = eprops.get("exec") or subjects.get(subj_ref.get(P+"UUID")) or "process"
        if et in TRAFFIC:
            ioc_ips = ips & WINDOWS[wid]["ips"]
            ip = min(ioc_ips) if ioc_ips else (min(ips) if ips else None)
            if ip:
                key = (proc, ip)
                if key not in chan:
                    chan[key] = {"proc": proc, "ip": ip, "count": 0, "mal": mal,
                                 "ts": e.get("timestampNanos"),
                                 "dir": "inbound" if et in INBOUND else "outbound"}
                chan[key]["count"] += 1
                if et in INBOUND:
                    chan[key]["dir"] = "inbound"
        elif et in SEMANTIC:
            ioc_paths = sorted(p for p in paths if path_matches_ioc(p, WINDOWS[wid]["paths"]))
            non_linker = sorted(p for p in paths if "ld-elf.so" not in p)
            tgt = (ioc_paths[0] if ioc_paths else
                   non_linker[0] if non_linker else
                   next(iter(sorted(paths)), None) or next(iter(sorted(ips)), None) or "a socket/pipe")
            sem.append({"type": et, "proc": proc, "target": tgt, "mal": mal, "ts": e.get("timestampNanos")})

    mal_sem = [x for x in sem if x["mal"]]
    ben_sem = [x for x in sem if not x["mal"]]
    mal_chan = [v for v in chan.values() if v["mal"]]
    ben_chan = [v for v in chan.values() if not v["mal"]]
    n_mal = len(mal_sem) + len(mal_chan)
    ben_keep = random.sample(ben_sem, min(len(ben_sem), max(n_mal * 10, 10)))
    ben_chan_keep = random.sample(ben_chan, min(len(ben_chan), max(n_mal, 5)))

    for x in mal_sem + ben_keep:
        text = f'Process {x["proc"]} {VERB.get(x["type"], x["type"])} {x["target"]}{role_tag(x["target"])}'
        alerts.append({"window": wid, "label": "malicious" if x["mal"] else "benign", "ts": x["ts"], "text": text})

    for c in mal_chan + ben_chan_keep:
        alerts.append({"window": wid, "label": "malicious" if c["mal"] else "benign", "ts": c["ts"],
                       "text": f'Process {c["proc"]} {chan_phrase(c)}'})

    print(f"{wid}: {n_mal} malicious, {len(ben_keep) + len(ben_chan_keep)} benign sampled")

alerts.sort(key=lambda a: (a["window"], a["ts"]))
texts = [a["text"] for a in alerts]
labels = [a["label"] for a in alerts]
awins = [a["window"] for a in alerts]
print(f"Total alerts: {len(alerts)}")

**3. Cluster alerts**

In [ ]:
from sentence_transformers import SentenceTransformer, util

print('Embedding alert texts...')
model = SentenceTransformer("all-MiniLM-L6-v2")
emb = model.encode(texts, convert_to_tensor=True, show_progress_bar=True)
print('Encoded', emb.shape)


In [ ]:
# Threshold sweep: evaluate clustering quality across similarity thresholds.
# Review the results below, then set BEST_THRESHOLD in the next cell.
import pandas as pd

def purity_per_window(comms, lbls, wins, window_ids):
    if not comms:
        return 0.0
    ps = []
    for members in comms:
        w_labels = [lbls[j] for j in members]
        majority = max(Counter(w_labels).values())
        ps.append(majority / len(members))
    return float(np.mean(ps)) if ps else 0.0

def intra_sim_per_window(comms, emb, max_sample=100):
    if not comms:
        return 0.0
    sims = []
    for members in comms:
        idx = members[:max_sample]
        if len(idx) < 2:
            continue
        m = util.pytorch_cos_sim(emb[idx], emb[idx])
        mask = np.triu(np.ones(m.shape, dtype=bool), k=1)
        sims.append(float(m.cpu().numpy()[mask].mean()))
    return float(np.mean(sims)) if sims else 0.0

sweep_results = []
for th in [0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80, 0.85, 0.90, 0.95]:
    all_comms = []
    n_clustered = 0
    for wid in WINDOWS:
        idx = [j for j, w in enumerate(awins) if w == wid]
        if len(idx) < 3:
            continue
        sub = util.community_detection(emb[idx], threshold=th, min_community_size=3)
        comms = [[idx[k] for k in c] for c in sub]
        all_comms.extend(comms)
        n_clustered += sum(len(c) for c in comms)

    n_comm = len(all_comms)
    pur = purity_per_window(all_comms, labels, awins, list(WINDOWS.keys()))
    sim = intra_sim_per_window(all_comms, emb)
    mal_dom = sum(1 for c in all_comms
                  if sum(1 for j in c if labels[j] == "malicious") > len(c) / 2)
    score = (pur * sim) if n_comm >= 3 else 0.0
    sweep_results.append({'threshold': th, 'n_comm': n_comm, 'clustered': n_clustered,
                          'mal_dominant': mal_dom, 'purity': round(pur, 4),
                          'intra_sim': round(sim, 4), 'score': round(score, 4)})

sweep_df = pd.DataFrame(sweep_results)
print('Threshold sweep:')
print(sweep_df.to_string(index=False))


In [ ]:
# Set threshold manually after reviewing sweep results above
BEST_THRESHOLD = 0.75
print(f'Using BEST_THRESHOLD = {BEST_THRESHOLD}')


In [ ]:
# Re-run clustering with the chosen threshold
communities = []
for wid in WINDOWS:
    idx = [j for j, w in enumerate(awins) if w == wid]
    if len(idx) < 3:
        continue
    sub = util.community_detection(emb[idx], threshold=BEST_THRESHOLD, min_community_size=3)
    communities += [[idx[k] for k in c] for c in sub]

def malicious_dominant(comms):
    return [i for i, c in enumerate(comms)
            if sum(1 for j in c if labels[j] == "malicious") > len(c) / 2]

MAL_COMMS = malicious_dominant(communities)
print(f'{len(communities)} communities | {len(MAL_COMMS)} malicious-dominant')
for cid in MAL_COMMS:
    comm = communities[cid]
    n_mal = sum(1 for j in comm if labels[j] == "malicious")
    win = Counter(awins[j] for j in comm).most_common(1)[0][0]
    print(f'  C{cid}: {len(comm)} alerts, {n_mal} malicious, mainly {win}')


**4. Extract triples**

In [ ]:
import re
from pathlib import Path
from llama_cpp import Llama, LlamaGrammar

if not os.path.isfile(MODEL_PATH):
    raise FileNotFoundError(
        f'Model file not found: {MODEL_PATH}\n'
        f'Download from: https://huggingface.co/bartowski/Mistral-Nemo-Instruct-2407-GGUF')

print(f'Loading LLM from {MODEL_PATH}...')
llm = Llama(model_path=MODEL_PATH, n_ctx=4096, n_gpu_layers=-1, seed=SEED, verbose=False)
print('LLM loaded')

def normalise_entity(text):
    text = re.sub(r'[^a-z0-9_]+', '_', text.lower().strip())
    return re.sub(r'_+', '_', text).strip('_')[:80]

HOST_RELATIONS = [
    'ACCESS_CREDENTIALS', 'EXPLOITS_VULNERABILITY', 'ESTABLISHES_C2',
    'MOVES_LATERALLY', 'EXFILTRATES_DATA', 'EXECUTES_PAYLOAD',
    'CHANGES_PERMISSIONS', 'DELETES_FILE', 'INJECTS_PROCESS', 'WRITES_FILE',
]

RELATION_TO_TECHNIQUES = {
    'ACCESS_CREDENTIALS': ['T1555', 'T1078', 'T1110'],
    'EXPLOITS_VULNERABILITY': ['T1190', 'T1203'],
    'ESTABLISHES_C2': ['T1071', 'T1071.001', 'T1071.004'],
    'MOVES_LATERALLY': ['T1021', 'T1570'],
    'EXFILTRATES_DATA': ['T1041', 'T1048'],
    'EXECUTES_PAYLOAD': ['T1059', 'T1059.007', 'T1105'],
    'CHANGES_PERMISSIONS': ['T1222', 'T1548'],
    'DELETES_FILE': ['T1070.004', 'T1070'],
    'INJECTS_PROCESS': ['T1055', 'T1055.001'],
    'WRITES_FILE': ['T1105'],
}

TRIPLE_SCHEMA = {
    'type': 'object',
    'properties': {
        'triples': {
            'type': 'array', 'minItems': 1, 'maxItems': 4,
            'items': {
                'type': 'object',
                'properties': {
                    'subject': {'type': 'string'},
                    'relation': {'type': 'string', 'enum': HOST_RELATIONS},
                    'target': {'type': 'string'},
                },
                'required': ['subject', 'relation', 'target'],
            },
        },
    },
    'required': ['triples'],
}
host_grammar = LlamaGrammar.from_json_schema(json.dumps(TRIPLE_SCHEMA))

RELATION_GUIDE = """Relation definitions (use exactly as written):
  EXPLOITS_VULNERABILITY : initial exploit of a service (malformed request to a web server)
  ESTABLISHES_C2         : outbound connection to an external command-and-control address
  EXECUTES_PAYLOAD       : running a dropped binary or command
  WRITES_FILE            : writing/dropping a file to disk
  CHANGES_PERMISSIONS    : changing file permissions (often to enable execution/elevation)
  INJECTS_PROCESS        : injecting code into another process
  DELETES_FILE           : removing a file (often to cover tracks)
  EXFILTRATES_DATA       : transferring data off the host"""

EXPLOIT_INBOUND_TECH = "T1190"
EXPLOIT_LOCAL_TECH = "T1203"

In [ ]:
def extract_triples(alert_texts, tag):
    block = "\n".join(f"- {t}" for t in alert_texts[:6])
    prompt = f"""[INST] You are a cybersecurity analyst analyzing host audit events.
Extract 1-4 semantic triples describing the attack behaviour.

{RELATION_GUIDE}

subject = process/actor; relation = the ONE best fit; target = file/address/process.
Name what you observe. No generic placeholders.

Example (illustrative FORMAT ONLY -- do NOT copy these names or relations):
{{"triples": [
  {{"subject":"some_process","relation":"WRITES_FILE","target":"some_file"}},
  {{"subject":"some_process","relation":"CHANGES_PERMISSIONS","target":"some_file"}}
]}}

ALERTS ({tag}):
{block}

Return ONLY valid JSON. [/INST]"""
    out = llm(prompt, max_tokens=512, temperature=0, seed=SEED,
              grammar=host_grammar, repeat_penalty=1.1, stop=["[/INST]"])
    try:
        triples = json.loads(out["choices"][0]["text"].strip()).get("triples", [])
    except Exception as e:
        print(f"  [{tag}] parse failed: {e}")
        triples = []
    valid = []
    for t in triples:
        s = normalise_entity(str(t.get("subject", "")))
        r = str(t.get("relation", "")).upper().strip()
        o = normalise_entity(str(t.get("target", "")))
        if not s or not o:
            continue
        if r not in HOST_RELATIONS:
            r = "EXECUTES_PAYLOAD"
        valid.append({"subject": s, "relation": r, "target": o})
    return valid

community_triples = {}
for cid in MAL_COMMS:
    member_texts = [texts[j] for j in communities[cid]]
    win = Counter(awins[j] for j in communities[cid]).most_common(1)[0][0]
    tr = extract_triples(member_texts, f"C{cid}")
    community_triples[cid] = {"triples": tr, "window": win}
    print(f"C{cid} ({win}):")
    for t in tr:
        print(f"  ({t['subject']}, {t['relation']}, {t['target']})")

**5. Map to ATT&CK**

In [ ]:
import numpy as np
from pathlib import Path
from sentence_transformers import util as sutil

if not (ATTCK_DIR / 'enterprise-attack.json').exists():
    raise FileNotFoundError(f'ATT&CK data not found at {ATTCK_DIR / "enterprise-attack.json"}')

def load_attck(path=ATTCK_DIR / 'enterprise-attack.json'):
    bundle = json.load(open(path))
    techs = {}
    for obj in bundle["objects"]:
        if obj.get("type") != "attack-pattern" or obj.get("revoked") or obj.get("x_mitre_deprecated"):
            continue
        tid = next((r["external_id"] for r in obj.get("external_references", [])
                    if r.get("source_name") == "mitre-attack"), None)
        if not tid:
            continue
        tac = [p["phase_name"] for p in obj.get("kill_chain_phases", [])
               if p.get("kill_chain_name") == "mitre-attack"]
        techs[tid] = {"name": obj.get("name", ""), "description": obj.get("description", ""),
                      "tactic": tac[0] if tac else ""}
    return techs

attck = load_attck()
if not attck:
    raise RuntimeError('No ATT&CK techniques loaded — check the enterprise-attack.json file')

tech_ids = list(attck.keys())
tech_texts = [f"{attck[t]['name']}. {attck[t]['description']}" for t in tech_ids]
tech_embs = model.encode(tech_texts, convert_to_tensor=True, show_progress_bar=True)
print(f"Embedded {len(tech_ids)} ATT&CK techniques")


In [ ]:
TOP_K = 5
MIN_REL_SUPPORT = 2

def parent(t):
    return t.split(".")[0] if t and t != "Unknown" else t

def parents(ts):
    return {parent(t) for t in ts}

def in_gt_set(gt_set, pred):
    if not pred or pred == "Unknown":
        return False
    pred_parent = pred.split(".")[0]
    return any(pred_parent == g.split(".")[0] for g in gt_set)

def retrieve(query, k=TOP_K):
    q = model.encode(query, convert_to_tensor=True)
    sims = sutil.pytorch_cos_sim(q, tech_embs)[0].cpu().numpy()
    return [tech_ids[i] for i in np.argsort(-sims)[:k]]

def kg_candidates(triples):
    rels = [t["relation"] for t in triples]
    if not rels:
        return []
    dom = Counter(rels).most_common(1)[0][0]
    return [t for t in RELATION_TO_TECHNIQUES.get(dom, []) if t in attck]

def community_has_inbound(cid):
    return any("accepted inbound connection" in texts[j] for j in communities[cid])

results = []
for cid in MAL_COMMS:
    info = community_triples[cid]
    triples, win = info["triples"], info["window"]
    gt_set = WINDOWS[win]["gt_set"]
    alert_text = " ".join(texts[j] for j in communities[cid])
    triple_text = " ".join(f"{t['subject']} {t['relation']} {t['target']}" for t in triples)
    cands = kg_candidates(triples)
    retrieved = retrieve(alert_text[:400] + " " + triple_text)
    dom_rel = Counter(t["relation"] for t in triples).most_common(1)[0][0] if triples else "none"

    resolved = None
    if dom_rel == "EXPLOITS_VULNERABILITY":
        resolved = EXPLOIT_INBOUND_TECH if community_has_inbound(cid) else EXPLOIT_LOCAL_TECH

    if resolved is not None:
        pred = resolved
    elif cands:
        q_emb = model.encode(alert_text[:400], convert_to_tensor=True)
        c_embs = tech_embs[[tech_ids.index(t) for t in cands]]
        sims = sutil.pytorch_cos_sim(q_emb, c_embs)[0].cpu().tolist()
        pred = cands[int(np.argmax(sims))]
    else:
        pred = retrieved[0] if retrieved else "Unknown"

    rel_counts = Counter(t["relation"] for t in triples)
    dom_relation = rel_counts.most_common(1)[0][0] if rel_counts else None
    kept_rels = [rel for rel, c in rel_counts.items() if c >= MIN_REL_SUPPORT or rel == dom_relation]
    pred_multi = sorted({next((x for x in RELATION_TO_TECHNIQUES.get(rel, []) if x in attck), None)
                         for rel in kept_rels} - {None})
    if community_has_inbound(cid):
        pred_multi = sorted(set(pred_multi) | {EXPLOIT_INBOUND_TECH})

    results.append({
        "cid": cid, "window": win, "gt_set": sorted(gt_set),
        "pred": pred, "pred_multi": pred_multi, "dom_rel": dom_rel,
        "parent": in_gt_set(gt_set, pred),
    })

print(f"{'C':>3} {'win':>4} {'pred':>9} {'dom_rel':>22} {'hit':>5}")
for r in results:
    print(f"{r['cid']:>3} {r['window']:>4} {r['pred']:>9} {r['dom_rel']:>22} {str(r['parent']):>5}")

def window_prf(get_techs):
    print(f"{'win':>4} {'P':>5} {'R':>5} {'F1':>5} {'T1190':>7}  predicted")
    macro = []
    for wid in WINDOWS:
        wr = [r for r in results if r["window"] == wid]
        G = parents(WINDOWS[wid]["gt_observable"])
        if not wr:
            print(f"{wid:>4} {'-':>5} {'-':>5} {'-':>5} {'-':>7}")
            macro.append((0.0, 0.0, 0.0))
            continue
        P = set()
        for r in wr:
            P |= parents(get_techs(r))
        tp = P & G
        prec = len(tp) / len(P) if P else 0.0
        rec = len(tp) / len(G) if G else 0.0
        f1 = 2 * prec * rec / (prec + rec) if (prec + rec) else 0.0
        macro.append((prec, rec, f1))
        t1190 = "yes" if "T1190" in P else "no"
        print(f"{wid:>4} {prec:>5.2f} {rec:>5.2f} {f1:>5.2f} {t1190:>7}  {sorted(P)}")
    mp, mr, mf = (sum(x[i] for x in macro) / len(macro) for i in range(3))
    print(f"macro: P={mp:.2f} R={mr:.2f} F1={mf:.2f}")

print("\nPer-window (multi-label, observable GT):")
window_prf(lambda r: r["pred_multi"])

mal_in_comm = {j for cid in MAL_COMMS for j in communities[cid] if labels[j] == "malicious"}
t1190_win = sum(1 for wid in WINDOWS
                if any("T1190" in parents(r["pred_multi"]) for r in results if r["window"] == wid))
print(f"\nMalicious alerts in clusters: {len(mal_in_comm)}")
print(f"Windows with T1190: {t1190_win}/{len(WINDOWS)}")

In [ ]:
# --- Baseline comparison: simple embedding retrieval (no KG, no relation mapping) ---
# For each community, retrieve the top-1 technique by direct cosine similarity to alert text.

baseline_results = []
for cid in MAL_COMMS:
    info = community_triples[cid]
    triples, win = info["triples"], info["window"]
    gt_set = WINDOWS[win]["gt_set"]
    alert_text = " ".join(texts[j] for j in communities[cid])
    triple_text = " ".join(f"{t['subject']} {t['relation']} {t['target']}" for t in triples)

    # Simple retrieval: no KG, no relation mapping, just raw embedding similarity
    query = (alert_text[:400] + " " + triple_text)[:512]
    q_emb = model.encode(query, convert_to_tensor=True)
    sims = sutil.pytorch_cos_sim(q_emb, tech_embs)[0].cpu().numpy()
    retrieved_order = [tech_ids[i] for i in np.argsort(-sims)]
    base_pred = retrieved_order[0] if retrieved_order else "Unknown"

    # Multi-label: top-K retrieved techniques
    base_pred_multi = retrieved_order[:TOP_K]

    baseline_results.append({
        "cid": cid, "window": win, "gt_set": sorted(gt_set),
        "pred": base_pred, "pred_multi": base_pred_multi,
        "parent": in_gt_set(gt_set, base_pred),
    })

# --- Bootstrap confidence intervals for per-window metrics ---
def bootstrap_window_ci(results_list, get_techs, n_boot=2000, seed=SEED):
    """Compute 95% CI for macro-F1 via bootstrap over windows."""
    rng = np.random.default_rng(seed)
    windows = list(WINDOWS.keys())
    boot_f1s = []
    for _ in range(n_boot):
        sampled = rng.choice(windows, size=len(windows), replace=True)
        f1s = []
        for wid in sampled:
            wr = [r for r in results_list if r["window"] == wid]
            G = parents(WINDOWS[wid]["gt_observable"])
            if not wr:
                continue
            P = set()
            for r in wr:
                P |= parents(get_techs(r))
            tp = P & G
            prec = len(tp) / len(P) if P else 0.0
            rec = len(tp) / len(G) if G else 0.0
            f1 = 2 * prec * rec / (prec + rec) if (prec + rec) else 0.0
            f1s.append(f1)
        boot_f1s.append(float(np.mean(f1s)) if f1s else 0.0)
    return [round(float(np.percentile(boot_f1s, 2.5)), 4),
            round(float(np.percentile(boot_f1s, 97.5)), 4)]

# Compute bootstrap CIs for both systems
kg_ci = bootstrap_window_ci(results, lambda r: r["pred_multi"])
base_ci = bootstrap_window_ci(baseline_results, lambda r: r["pred_multi"])

# --- Print comparison table ---
print("=" * 65)
print("  System Comparison: KG-based (RAG) vs Simple Retrieval Baseline")
print("=" * 65)

def print_comparison(results_list, get_techs, label, ci):
    print(f"\n--- {label} ---")
    print(f"{'win':>4}  {'P':>5} {'R':>5} {'F1':>5}  predicted")
    macro = []
    for wid in WINDOWS:
        wr = [r for r in results_list if r["window"] == wid]
        G = parents(WINDOWS[wid]["gt_observable"])
        if not wr:
            print(f"{wid:>4}  {'-':>5} {'-':>5} {'-':>5}")
            macro.append((0.0, 0.0, 0.0))
            continue
        P = set()
        for r in wr:
            P |= parents(get_techs(r))
        tp = P & G
        prec = len(tp) / len(P) if P else 0.0
        rec = len(tp) / len(G) if G else 0.0
        f1 = 2 * prec * rec / (prec + rec) if (prec + rec) else 0.0
        macro.append((prec, rec, f1))
        print(f"{wid:>4}  {prec:>5.2f} {rec:>5.2f} {f1:>5.2f}  {sorted(P)}")
    mp, mr, mf = (sum(x[i] for x in macro) / len(macro) for i in range(3))
    print(f"macro: P={mp:.2f} R={mr:.2f} F1={mf:.2f}  CI95(F1)=[{ci[0]:.4f},{ci[1]:.4f}]")
    return mp, mr, mf

kg_p, kg_r, kg_f1 = print_comparison(results, lambda r: r["pred_multi"], "KG-based (RAG)", kg_ci)
base_p, base_r, base_f1 = print_comparison(baseline_results, lambda r: r["pred_multi"], "Simple Retrieval Baseline", base_ci)

print(f"\n{'':>4}  {'P':>5} {'R':>5} {'F1':>5}")
print(f"Delta (KG - Baseline): {kg_p-base_p:>5.2f} {kg_r-base_r:>5.2f} {kg_f1-base_f1:>5.2f}")

# --- Store comparison metrics ---
comparison_metrics = {
    "kg_macro_precision": round(kg_p, 4),
    "kg_macro_recall": round(kg_r, 4),
    "kg_macro_f1": round(kg_f1, 4),
    "kg_f1_ci95": kg_ci,
    "baseline_macro_precision": round(base_p, 4),
    "baseline_macro_recall": round(base_r, 4),
    "baseline_macro_f1": round(base_f1, 4),
    "baseline_f1_ci95": base_ci,
    "delta_precision": round(kg_p - base_p, 4),
    "delta_recall": round(kg_r - base_r, 4),
    "delta_f1": round(kg_f1 - base_f1, 4),
}
print(f"\nComparison metrics: {json.dumps(comparison_metrics, indent=2)}")


**6. Save**

In [ ]:
import torch, os

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

state = {
    "seed": SEED,
    "best_threshold": BEST_THRESHOLD,
    "alerts": alerts,
    "communities": [[int(j) for j in c] for c in communities],
    "mal_comms": MAL_COMMS,
    "community_triples": {str(k): v for k, v in community_triples.items()},
    "results": results,
    "baseline_results": baseline_results,
}
with open(RESULTS_DIR / "darpa_cadets_run.json", "w") as f:
    json.dump(state, f, indent=2)
torch.save(emb, RESULTS_DIR / "darpa_cadets_emb.pt")

# Save comparison and sweep metrics
with open(RESULTS_DIR / "darpa_comparison_metrics.json", "w") as f:
    json.dump(comparison_metrics, f, indent=2)
sweep_df.to_csv(RESULTS_DIR / "darpa_threshold_sweep.csv", index=False)

print("Saved to", RESULTS_DIR)

**7. Visualizations**

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

(RESULTS_DIR / 'figures').mkdir(parents=True, exist_ok=True)

In [ ]:
# Figure 1: Threshold sweep plot
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].plot(sweep_df['threshold'], sweep_df['n_comm'], 'bo-', markersize=4)
axes[0].set_xlabel('Threshold'); axes[0].set_ylabel('Number of communities')
axes[0].set_title('Community count vs threshold'); axes[0].grid(True, alpha=0.3)

axes[1].plot(sweep_df['threshold'], sweep_df['purity'], 'go-', markersize=4)
axes[1].set_xlabel('Threshold'); axes[1].set_ylabel('Purity (malicious vs benign)')
axes[1].set_title('Purity vs threshold'); axes[1].grid(True, alpha=0.3)
axes[1].set_ylim(0, 1)

axes[2].plot(sweep_df['threshold'], sweep_df['score'], 'ro-', markersize=4)
axes[2].axvline(x=BEST_THRESHOLD, color='gray', linestyle='--', alpha=0.7, label=f'Best={BEST_THRESHOLD}')
axes[2].set_xlabel('Threshold'); axes[2].set_ylabel('Combined score')
axes[2].set_title('Score vs threshold'); axes[2].grid(True, alpha=0.3)
axes[2].legend()

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'figures' / 'darpa_threshold_sweep.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: darpa_threshold_sweep.png")

In [ ]:
# Figure 2: Per-window precision/recall/F1 comparison
windows = sorted(WINDOWS.keys())

def extract_prf(results_list, get_techs):
    precs, recs, f1s = [], [], []
    for wid in windows:
        wr = [r for r in results_list if r["window"] == wid]
        G = parents(WINDOWS[wid]["gt_observable"])
        if not wr:
            precs.append(0.0); recs.append(0.0); f1s.append(0.0)
            continue
        P = set()
        for r in wr:
            P |= parents(get_techs(r))
        tp = P & G
        precs.append(len(tp) / len(P) if P else 0.0)
        recs.append(len(tp) / len(G) if G else 0.0)
        f1 = 2 * precs[-1] * recs[-1] / (precs[-1] + recs[-1]) if (precs[-1] + recs[-1]) else 0.0
        f1s.append(f1)
    return precs, recs, f1s

kg_p, kg_r, kg_f1 = extract_prf(results, lambda r: r["pred_multi"])
base_p, base_r, base_f1 = extract_prf(baseline_results, lambda r: r["pred_multi"])

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
x = np.arange(len(windows))
w = 0.35

for ax, kg_vals, base_vals, title, ylabel in [
    (axes[0], kg_p, base_p, 'Precision', 'Precision'),
    (axes[1], kg_r, base_r, 'Recall', 'Recall'),
    (axes[2], kg_f1, base_f1, 'F1 Score', 'F1 Score')]:
    ax.bar(x - w/2, kg_vals, w, label='KG-based (RAG)', color='steelblue')
    ax.bar(x + w/2, base_vals, w, label='Simple retrieval', color='lightcoral')
    ax.set_xticks(x); ax.set_xticklabels(windows)
    ax.set_title(title); ax.set_ylabel(ylabel); ax.legend()
    ax.grid(True, alpha=0.3, axis='y'); ax.set_ylim(0, 1)

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'figures' / 'darpa_per_window_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: darpa_per_window_comparison.png")

In [ ]:
# Figure 3: Summary bar chart
fig, ax = plt.subplots(figsize=(8, 5))
systems = ['KG-based (RAG)', 'Simple Retrieval', 'Delta']
macros = {
    'Precision': [comparison_metrics['kg_macro_precision'],
                  comparison_metrics['baseline_macro_precision'],
                  comparison_metrics['delta_precision']],
    'Recall': [comparison_metrics['kg_macro_recall'],
               comparison_metrics['baseline_macro_recall'],
               comparison_metrics['delta_recall']],
    'F1': [comparison_metrics['kg_macro_f1'],
           comparison_metrics['baseline_macro_f1'],
           comparison_metrics['delta_f1']],
}
x = np.arange(len(systems))
w = 0.25
colors = ['steelblue', 'lightcoral', 'gray']
for i, (metric, vals) in enumerate(macros.items()):
    offset = (i - 1) * w
    bars = ax.bar(x + offset, vals, w, label=metric, color=colors, alpha=0.8)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                f'{val:.2f}', ha='center', va='bottom', fontsize=8)
ax.set_xticks(x); ax.set_xticklabels(systems)
ax.set_ylabel('Score'); ax.set_title('DARPA CADETS E3: System Comparison')
ax.legend(); ax.grid(True, alpha=0.3, axis='y'); ax.set_ylim(0, 1)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'figures' / 'darpa_metrics_summary.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: darpa_metrics_summary.png")

In [ ]:
# Figure 4: Malicious alerts in clusters
fig, ax = plt.subplots(figsize=(8, 4))
mal_per_win = {}
for wid in windows:
    mal_per_win[wid] = len({j for cid in MAL_COMMS
                             for j in communities[cid]
                             if labels[j] == "malicious" and awins[j] == wid})
total_per_win = {}
for wid in windows:
    total_per_win[wid] = sum(1 for j, w in enumerate(awins) if w == wid)

x = np.arange(len(windows))
ax.bar(x, [total_per_win[w] for w in windows], label='Total alerts', color='lightgray')
ax.bar(x, [mal_per_win[w] for w in windows], label='Malicious in clusters', color='crimson')
ax.set_xticks(x); ax.set_xticklabels(windows)
ax.set_ylabel('Alert count'); ax.set_title('Alerts per window (DARPA CADETS E3)')
ax.legend(); ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'figures' / 'darpa_alerts_per_window.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved: darpa_alerts_per_window.png")

print(f"All figures saved to {RESULTS_DIR / 'figures'}")